# Trees and Model Stability

Trees are notorious for being **unstable**: Small changes in the data can lead to noticeable or large changes in the tree. We're going to explore this phenomenon.

1. Check out the data set `heart_failure` and appropriately clean it.
2. Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data and evaluate their performance on the test data. Set aside the predictions these models make.
3. Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.
4. Typically, you would see the trees changing what appears to be a non-trivial amount, while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial. 
But are they? Instead of focusing on the tree or model coefficients, do three things:
    
    a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?
    
    b. Compute the correlation between your model in part 2 and your alternative models in part 3, separately for your trees and linear models. Are they highly correlated or not?
    
    c. Run a simple linear regression of the predicted values on the test set from the alternative models on the predicted values from question 2, separately for your trees and linear models. Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?

5. Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?
6. True or false for this dataset, and explain: "Even if the models end up having a substantially different appearance, the predictions they generate are often very similar."

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [Sissi Wang, Alex Tan, Sadat Hassan]

**Date:** [11/18/2026]

In [ ]:
# Your Phase 1 solution to the problem goes here.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, root_mean_squared_error, r2_score, log_loss
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/heart_failure/heart_failure_clinical_records_dataset.csv")

# print(df.describe())
# print(df.info())

# Look at data types and shape
print("df.shape:")
print(df.shape, '\n')
print("df.dtypes:")
print(df.dtypes, '\n')
df.head()

# Check for nulls
print(df.isnull().sum(), "\n\n")

# Check for duplicates
print("Duplicates:", df.duplicated().sum())


# Scale and log transform the data
# Log transform heavily right-skewed features (stateless, so safe before the split)
skewed_cols = ["creatinine_phosphokinase", "serum_creatinine", "platelets"]
for col in skewed_cols:
    df[col] = np.log1p(df[col])

print(df.describe())

# Step 2: Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data
# and evaluate their performance on the test data. Set aside the predictions these models make.

y = df["DEATH_EVENT"]
X = df.drop(columns="DEATH_EVENT")
seed_base = 100

X_train_100, X_test_100, y_train_100, y_test_100 = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed_base,
    stratify = y
)

print(f"{len(X_train_100)} training patients; {len(X_test_100)} test patients")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_100)
X_test_scaled = scaler.transform(X_test_100)

# Decision Tree
tree_model = DecisionTreeClassifier(random_state=seed_base)
tree_model.fit(X_train_100, y_train_100)

# Logistic Regression Model
log_reg_model = LogisticRegression(C=np.inf, max_iter=5000, tol=1e-8)
log_reg_model.fit(X_train_scaled, y_train_100)

tree_probs = tree_model.predict_proba(X_test_100)[:, 1]
tree_preds = tree_model.predict(X_test_100)

log_reg_probs = log_reg_model.predict_proba(X_test_scaled)[:, 1]
log_reg_preds = log_reg_model.predict(X_test_scaled)

print("Decision Tree - Test Accuracy:", accuracy_score(y_test_100, tree_preds))
print("Decision Tree - Test BCE (Log Loss):", log_loss(y_test_100, tree_probs))

print("Logistic Regression - Test Accuracy:", accuracy_score(y_test_100, log_reg_preds))
print("Logistic Regression - Test BCE (Log Loss): ",log_loss(y_test_100, log_reg_probs))

# Step 3: Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.

# Load data & set up seeds to test
seeds = [42, 100, 202, 350, 512]

# Dictionaries to store results for comparison
results = {}
all_tree_models = {}
all_log_reg_models = {}
all_predictions = {}

for seed in seeds:
    # Perform train-test split with current seed
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    # Scale features for Logistic Regression
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Fit Decision Tree
    tree_model = DecisionTreeClassifier(random_state=seed)
    tree_model.fit(X_train, y_train)

    # Fit Logistic Regression Model
    log_reg_model = LogisticRegression(C=np.inf, max_iter=5000, tol=1e-8)
    log_reg_model.fit(X_train_scaled, y_train)

    # Generate and set aside predictions
    tree_probs = tree_model.predict_proba(X_test)[:, 1]
    tree_preds = tree_model.predict(X_test)

    log_reg_probs = log_reg_model.predict_proba(X_test_scaled)[:, 1]
    log_reg_preds = log_reg_model.predict(X_test_scaled)

    # Store predictions and models
    all_predictions[seed] = {
        "y_test": y_test,
        "tree_preds": tree_preds,
        "tree_probs": tree_probs,
        "log_reg_preds": log_reg_preds,
        "log_reg_probs": log_reg_probs
    }
    all_tree_models[seed] = tree_model
    all_log_reg_models[seed] = log_reg_model

    # Record metrics & coefficients
    results[seed] = {
        "Tree Accuracy": accuracy_score(y_test, tree_preds),
        "Tree LogLoss": log_loss(y_test, tree_probs),
        "LogReg Accuracy": accuracy_score(y_test, log_reg_preds),
        "LogReg LogLoss": log_loss(y_test, log_reg_probs),
        "LogReg Coefficients": log_reg_model.coef_[0]
    }

# Compare How Much Models Vary Across Seeds
for seed, res in results.items():
    print("\nSeed:", seed)
    print(f"Tree - Accuracy: {res['Tree Accuracy']:.3f} | LogLoss: {res['Tree LogLoss']:.3f}")
    print(f"LogReg - Accuracy: {res['LogReg Accuracy']:.3f} | LogLoss: {res['LogReg LogLoss']:.3f}")
    print(f"LogReg Coefficients: {np.round(res['LogReg Coefficients'], 3)}")

# Comparing Coefficient Stability vs Tree Stability
# Extracting coefficients into a dataframe for easy comparison
coef_df = pd.DataFrame([res["LogReg Coefficients"] for res in results.values()],
                       index=seeds, columns=X.columns)
print("Logistic Regression Coefficients Across Seeds")
print(coef_df.round(3))

##Typically, you would see the trees changing what appears to be a non-trivial amount,
#while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial.
#But are they? Instead of focusing on the tree or model coefficients, do three things:

seed_base = 100
alt_seeds = [42, 202, 350, 512]

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed_base,
    stratify=y
)

tree_predictions = {}
logit_predictions = {}
for s in [seed_base] + alt_seeds:
    X_train_s, _, _, _ = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=s,
        stratify=y
    )
    scaler_s = StandardScaler().fit(X_train_s)
    tree_predictions[s] = all_tree_models[s].predict_proba(X_test_base)[:, 1]
    logit_predictions[s] = all_log_reg_models[s].predict_proba(scaler_s.transform(X_test_base))[:, 1]

#a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values
#for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?


fig, axes = plt.subplots(2, len(alt_seeds), figsize=(14, 6.4))
for j, s in enumerate(alt_seeds):
    for i, (name, preds) in enumerate([("Tree", tree_predictions),
                                       ("Logistic regression", logit_predictions)]):
        plot_df = pd.DataFrame({"base": preds[seed_base], "alt": preds[s]})
        sns.scatterplot(data=plot_df, x="base", y="alt", s=25, ax=axes[i, j])
        axes[i, j].plot([0, 1], [0, 1], color="gray", linestyle="--")   # equality line
        axes[i, j].set_title(f"{name}: seed {s} vs seed {seed_base}")
        axes[i, j].set_xlabel(f"Predicted probability (seed {seed_base})")
        axes[i, j].set_ylabel(f"Predicted probability (seed {s})")
plt.tight_layout(); plt.show()

#4A ANSWER: the logistic regression points are close to the equality line, meaning those models are quite similar to each other
#The tree part however does not align with this as many are off the diagonal corners. For example, there are dots at 0,1 and 1,0
# these points are off the diagonal line itself.

#b. Compute the correlation between your model in part 2 and your alternative models in part 3,
#separately for your trees and linear models. Are they highly correlated or not?
#c. Run a simple linear regression of the predicted values on the test set from the alternative models on
#the predicted values from question 2, separately for your trees and linear models.
#Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?

rows = []
for name, preds in [("Tree", tree_predictions), ("Logistic regression", logit_predictions)]:
    for s in alt_seeds:
        base_p = pd.Series(preds[seed_base], name="base")
        alt_p = pd.Series(preds[s], name="alt")
        fit = LinearRegression().fit(base_p.to_frame(), alt_p)   # alt predictions on base predictions
        rows.append({
            "Model": name,
            "Alt seed": s,
            "Correlation": base_p.corr(alt_p),
            "Intercept": fit.intercept_,
            "Slope": fit.coef_[0],
            "R-squared": fit.score(base_p.to_frame(), alt_p),
            "Mean abs gap": np.abs(base_p - alt_p).mean(),
        })
q4_summary = pd.DataFrame(rows)
display(q4_summary.round(3))

#4B ANSWER: Logistic regression is 0.96 to 0.97 meaning it is highly correlated
#Whereas trees are 0.47 to 0.55 which is much less correlated.

#4C ANSWER: The logistic regression models have intercepts close to zero (< 0.05), slopes close to 1 (approximately 0.92-0.96),
# and R-squared values close to 1 (approximately 0.92–0.94) as shown in the table.

#Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?

display(coef_df.round(3))
display(coef_df.agg(["mean", "std"]).round(3))

tree_summary = pd.DataFrame({
    "Split seed": seeds,
    "Root feature": [
        X.columns[all_tree_models[s].tree_.feature[0]]
        for s in seeds
    ],

    "Leaves": [
        all_tree_models[s].get_n_leaves()
        for s in seeds
    ],

    "Test accuracy": [
        results[s]["Tree Accuracy"]
        for s in seeds
    ],
})
display(tree_summary)

#5 ANSWER:
# Linear models have relatively similar coefficients and predictions across the
# different train/test splits since their coefficients do not vary a lot, and
# their predictions are highly correlated.
# Trees, on the other hand, do not have similar coefficients and predictions as
# their structures and predictions vary more depending on the training sample
# (different seeds). Lower correlations shows that the trees are more sensitive
# to changes in the training data.

#True or false for this dataset, and explain: "Even if the models end up having a
#substantially different appearance, the predictions they generate are often very similar."


#6 ANSWER: The statement is partially true for this dataset, depending on the type of model being used.

# The statement is true for the logistic regressions since it produced very similar predictions
# across various train-test splits, with correlations of around 0.96-0.97.
# This is not the case, however, for the decision tress. The preictions we got from decision
# trees with various random seeds are much less correlated, with correlations of around 0.46-0.55.

# Although models with very different apperances may lead to similar predictions,
# it is more evident in logistic regression than in decision tress for this dataset.

df.shape:
(299, 13) 

df.dtypes:
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object 

age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64 


Duplicates: 0
              age     anaemia  creatinine_phosphokinase    diab

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]